In [2]:
import pandas as pd

from scipy.sparse import hstack
from sklearn.preprocessing import OneHotEncoder, normalize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

In [3]:
articles = pd.read_parquet("../../../data/gold/articles.parquet")

articles.shape

(105542, 15)

In [4]:
categorical_cols = [
    "product_type_name",
    "product_group_name",
    "graphical_appearance_name",
    "colour_group_name",
    "perceived_colour_value_name",
    "perceived_colour_master_name",
    "department_name",
    "index_name",
    "index_group_name",
    "section_name",
    "garment_group_name"
]

In [5]:
articles["combined_text"] = (
    articles["prod_name"].astype(str) + " " +
    articles["detail_desc"].astype(str)
)

In [6]:
categorical_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

categorical_matrix = categorical_encoder.fit_transform(
    articles[categorical_cols]
)

categorical_matrix.shape

(105542, 600)

In [7]:
tfidf_vectorizer = TfidfVectorizer(
    stop_words="english",
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    max_features=50000
)

text_matrix = tfidf_vectorizer.fit_transform(
    articles["combined_text"]
)

text_matrix.shape

(105542, 50000)

In [8]:
categorical_matrix_norm = normalize(
    categorical_matrix,
    norm="l2",
    axis=1
)

text_matrix_norm = normalize(
    text_matrix,
    norm="l2",
    axis=1
)

In [9]:
categorical_weight = 1.0
text_weight = 1.0

hybrid_matrix = hstack([
    categorical_matrix_norm * categorical_weight,
    text_matrix_norm * text_weight
])

hybrid_matrix.shape

(105542, 50600)

In [10]:
nn_hybrid_model = NearestNeighbors(
    metric="cosine",
    algorithm="brute"
)

nn_hybrid_model.fit(hybrid_matrix)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [11]:
def get_similar_products_hybrid(article_id, top_k=10, candidate_pool=200):
    matching_indices = articles.index[
        articles["article_id"] == article_id
    ].tolist()

    if not matching_indices:
        raise ValueError(f"article_id bulunamadı: {article_id}")

    query_idx = matching_indices[0]
    query_product_code = articles.loc[query_idx, "product_code"]

    distances, indices = nn_hybrid_model.kneighbors(
        hybrid_matrix[query_idx],
        n_neighbors=candidate_pool
    )

    filtered_neighbors = []
    seen_product_codes = set()

    for idx, dist in zip(indices[0], distances[0]):
        candidate_product_code = articles.loc[idx, "product_code"]

        # Kendisini çıkar
        if idx == query_idx:
            continue

        # Sorgu ürününün farklı renk varyantlarını çıkar
        if candidate_product_code == query_product_code:
            continue

        # Aynı önerilen ürün ailesinin farklı renklerini tekrar gösterme
        if candidate_product_code in seen_product_codes:
            continue

        filtered_neighbors.append((idx, dist))
        seen_product_codes.add(candidate_product_code)

        if len(filtered_neighbors) == top_k:
            break

    neighbor_indices = [idx for idx, _ in filtered_neighbors]
    neighbor_distances = [dist for _, dist in filtered_neighbors]

    display_cols = [
        "article_id",
        "product_code",
        "prod_name",
        "product_type_name",
        "colour_group_name",
        "section_name",
        "garment_group_name"
    ]

    query_product = articles.iloc[[query_idx]][display_cols]

    recommendations = articles.iloc[neighbor_indices][display_cols].copy()
    recommendations["cosine_similarity"] = 1 - pd.Series(
        neighbor_distances,
        index=recommendations.index
    )

    return query_product, recommendations

In [12]:
query_product, recommendations = get_similar_products_hybrid(
    article_id=108775015,
    top_k=10
)

query_product, recommendations

(   article_id  product_code  prod_name product_type_name colour_group_name  \
 0   108775015        108775  Strap top          Vest top             Black   
 
              section_name garment_group_name  
 0  Womens Everyday Basics       Jersey Basic  ,
         article_id  product_code                   prod_name  \
 10109    538699001        538699            V-neck strap top   
 83527    812371001        812371            Strap top 3-pack   
 72495    767869001        767869           V-neck Strap Top.   
 62638    736870001        736870            Strap Top 2 pack   
 47253    688463001        688463             Straptop 2-pack   
 102303   903309001        903309         V-neck straptop 3-p   
 94767    863937010        863937                 Vanessa rib   
 86782    824999001        824999                Strap top 2p   
 83574    812525001        812525  V-neck strap top 2-pack(1)   
 101064   893994001        893994  V-Neck Strap Top Long 2-pk   
 
        product_type_name 

In [13]:
test_article_ids = [
    108775015,  # Strap top
    681107007,  # Dress
    686284001,  # Sweater
    754256001,  # Bra
    755712001   # Shirt
]

for article_id in test_article_ids:
    query_product, recommendations = get_similar_products_hybrid(
        article_id=article_id,
        top_k=5
    )

    print("\nQUERY PRODUCT")
    display(query_product)

    print("RECOMMENDATIONS")
    display(recommendations)


QUERY PRODUCT


,article_id,product_code,prod_name,product_type_name,colour_group_name,section_name,garment_group_name
0,108775015,108775,Strap top,Vest top,Black,Womens Everyday Basics,Jersey Basic


RECOMMENDATIONS


,article_id,product_code,prod_name,product_type_name,colour_group_name,section_name,garment_group_name,cosine_similarity
10109,538699001,538699,V-neck strap top,Vest top,Black,Womens Everyday Basics,Jersey Basic,0.828563
83527,812371001,812371,Strap top 3-pack,Vest top,Black,Womens Everyday Basics,Jersey Basic,0.705904
72495,767869001,767869,V-neck Strap Top.,Vest top,Black,Womens Everyday Basics,Jersey Basic,0.700823
62638,736870001,736870,Strap Top 2 pack,Vest top,Black,Womens Everyday Basics,Jersey Basic,0.698167
47253,688463001,688463,Straptop 2-pack,Vest top,Black,Womens Everyday Basics,Jersey Basic,0.654173



QUERY PRODUCT


...

,article_id,product_code,prod_name,product_type_name,colour_group_name,section_name,garment_group_name,cosine_similarity
64525,742079001,742079,Panorama mid support bra,Bra,Black,Ladies H&M Sport,Jersey Fancy,0.810826
56020,712711001,712711,Greenville medium support bra,Bra,Black,Ladies H&M Sport,Jersey Fancy,0.792139
34107,640552001,640552,GREENVILLE med support spor,Bra,Black,Ladies H&M Sport,Jersey Fancy,0.773827
3789,466381012,466381,Greenville bra (1),Bra,Black,Ladies H&M Sport,Jersey Fancy,0.756981
64293,741087001,741087,Carina high support bra,Bra,Black,Ladies H&M Sport,Jersey Fancy,0.755202



QUERY PRODUCT


,article_id,product_code,prod_name,product_type_name,colour_group_name,section_name,garment_group_name
68888,755712001,755712,Sune slipover set,Shirt,White,Kids Boy,Shirts


RECOMMENDATIONS


,article_id,product_code,prod_name,product_type_name,colour_group_name,section_name,garment_group_name,cosine_similarity
96987,872483001,872483,Sune WD shirt (TVP),Shirt,White,Kids Boy,Shirts,0.745422
72211,767017002,767017,SU Odessa Easy Iron w bow/tie,Shirt,White,Kids Boy,Shirts,0.716176
66557,748438009,748438,Kent ss Shirt w bow,Shirt,White,Kids Boy,Shirts,0.716039
49014,693675002,693675,Sune shirt,Shirt,White,Kids Boy,Shirts,0.709573
21079,592039001,592039,Penguin shirt 2SET,Shirt,White,Kids Boy,Shirts,0.705751


In [14]:
categorical_weight = 1.5
text_weight = 1.0

hybrid_matrix = hstack([
    categorical_matrix_norm * categorical_weight,
    text_matrix_norm * text_weight
])

hybrid_matrix.shape

(105542, 50600)

In [15]:
nn_hybrid_model = NearestNeighbors(
    metric="cosine",
    algorithm="brute"
)

nn_hybrid_model.fit(hybrid_matrix)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [16]:
test_article_ids = [
    108775015,
    681107007,
    686284001,
    754256001,
    755712001
]

for article_id in test_article_ids:
    query_product, recommendations = get_similar_products_hybrid(
        article_id=article_id,
        top_k=5
    )

    print("\nQUERY PRODUCT")
    display(query_product)

    print("RECOMMENDATIONS")
    display(recommendations)


QUERY PRODUCT


,article_id,product_code,prod_name,product_type_name,colour_group_name,section_name,garment_group_name
0,108775015,108775,Strap top,Vest top,Black,Womens Everyday Basics,Jersey Basic


RECOMMENDATIONS


,article_id,product_code,prod_name,product_type_name,colour_group_name,section_name,garment_group_name,cosine_similarity
10109,538699001,538699,V-neck strap top,Vest top,Black,Womens Everyday Basics,Jersey Basic,0.894501
83527,812371001,812371,Strap top 3-pack,Vest top,Black,Womens Everyday Basics,Jersey Basic,0.819018
72495,767869001,767869,V-neck Strap Top.,Vest top,Black,Womens Everyday Basics,Jersey Basic,0.815891
62638,736870001,736870,Strap Top 2 pack,Vest top,Black,Womens Everyday Basics,Jersey Basic,0.814257
47254,688463003,688463,Straptop 2-pack,Vest top,Black,Womens Everyday Basics,Jersey Basic,0.787183



QUERY PRODUCT


...

,article_id,product_code,prod_name,product_type_name,colour_group_name,section_name,garment_group_name,cosine_similarity
64525,742079001,742079,Panorama mid support bra,Bra,Black,Ladies H&M Sport,Jersey Fancy,0.883585
56020,712711001,712711,Greenville medium support bra,Bra,Black,Ladies H&M Sport,Jersey Fancy,0.872086
34107,640552001,640552,GREENVILLE med support spor,Bra,Black,Ladies H&M Sport,Jersey Fancy,0.860817
3789,466381012,466381,Greenville bra (1),Bra,Black,Ladies H&M Sport,Jersey Fancy,0.850450
64293,741087001,741087,Carina high support bra,Bra,Black,Ladies H&M Sport,Jersey Fancy,0.849355



QUERY PRODUCT


,article_id,product_code,prod_name,product_type_name,colour_group_name,section_name,garment_group_name
68888,755712001,755712,Sune slipover set,Shirt,White,Kids Boy,Shirts


RECOMMENDATIONS


,article_id,product_code,prod_name,product_type_name,colour_group_name,section_name,garment_group_name,cosine_similarity
96987,872483001,872483,Sune WD shirt (TVP),Shirt,White,Kids Boy,Shirts,0.843337
72210,767017001,767017,SU Odessa Easy Iron w bow/tie,Shirt,White,Kids Boy,Shirts,0.825339
66553,748438002,748438,Kent ss Shirt w bow,Shirt,White,Kids Boy,Shirts,0.825255
49014,693675002,693675,Sune shirt,Shirt,White,Kids Boy,Shirts,0.821276
21079,592039001,592039,Penguin shirt 2SET,Shirt,White,Kids Boy,Shirts,0.818924


## Initial Findings

The hybrid similarity baseline combines categorical metadata and TF-IDF text features.

Compared to the categorical-only baseline, the hybrid model improves ranking within identical or highly similar category groups by using product names and descriptions. This helps distinguish products that share the same categorical profile but differ in style, naming, pack structure, or product-specific wording.

Compared to the text-only baseline, the hybrid model reduces category drift. Generic terms such as `strap`, `set`, or seasonal product words are less likely to pull unrelated products into the recommendation list when categorical metadata is also included.

The first unweighted hybrid version already improved the recommendation quality. However, increasing the categorical feature weight produced more reliable recommendations by keeping results closer to the original product family.

Current selected baseline:

- `categorical_weight = 1.5`
- `text_weight = 1.0`

The recommendation function also applies product-level filtering:

- removes the query item itself
- removes other colour variants of the same product using `product_code`
- keeps only one result per recommended `product_code`

This prevents the recommendation list from being filled with different colour variants of the same product.

Although category drift should be controlled, small variations in product type are not always harmful. In some cases, related product types can improve discovery and provide useful alternatives. Therefore, the current baseline does not strictly filter by `product_type_name`; instead, it uses weighted categorical features to keep recommendations close while still allowing limited diversity.

Overall, this hybrid approach is the strongest baseline so far because it keeps recommendations in the correct product family while still using text features for fine-grained ranking.